In [1]:
import arcpy
import os

In [2]:
workspace = arcpy.env.workspace

In [3]:
# Create raster objects for each climate scenario from the service URLs

TemperatureMultidimensional = arcpy.Raster("https://tiledimageservices3.arcgis.com/0Fs3HcaFfvzXvm7w/arcgis/rest/services/LOCA2_Temperature_Projections/ImageServer")
Precip370Multidimensional = arcpy.Raster("https://tiledimageservices3.arcgis.com/0Fs3HcaFfvzXvm7w/arcgis/rest/services/LOCA2_v2_Precip_370/ImageServer")
Precip245Multidimensional = arcpy.Raster("https://tiledimageservices3.arcgis.com/0Fs3HcaFfvzXvm7w/arcgis/rest/services/LOCA2_v2_Precip_245/ImageServer")
SLR_EC_Int = arcpy.Raster("https://tiledimageservices3.arcgis.com/0Fs3HcaFfvzXvm7w/arcgis/services/CMRA_SLR_Inundation_RCP45_Early_2030/ImageServer")
SLR_LC_Intermediate = arcpy.Raster("https://tiledimageservices3.arcgis.com/0Fs3HcaFfvzXvm7w/arcgis/services/CMRA_SLR_Inundation_RCP45_Late_2090/ImageServer")
SLR_LC_IntHigh = arcpy.Raster("https://tiledimageservices3.arcgis.com/0Fs3HcaFfvzXvm7w/arcgis/services/CMRA_SLR_Inundation_RCP85_Late_2090/ImageServer")
WildfireHazardPotential = arcpy.Raster("https://imagery.geoplatform.gov/iipp/services/Fire_Aviation/USFS_EDW_RMRS_WildfireHazardPotentialClassified/ImageServer")

In [29]:
# Create variables representing each of the CSV tables, downloaded from https://www.imls.gov/sites/default/files/2018_csv_museum_data_files.zip and unzipped
aprx = arcpy.mp.ArcGISProject("CURRENT")
data_dir = os.path.join(os.path.dirname(aprx.filePath))

MuseumCSV1 = os.path.join(data_dir, "MuseumFile2018_File1_Nulls.csv")
MuseumCSV2 = os.path.join(data_dir, "MuseumFile2018_File2_Nulls.csv")
MuseumCSV3 = os.path.join(data_dir, "MuseumFile2018_File3_Nulls.csv")

#Note: Blank row in row 2 of MuseumFile2018_File3_Nulls.csv must be deleted before continuing

In [32]:
#Creating layers from each Museum CSV using the LONGITUDE AND LATITUDE fields, then combining all three layers into one Feature Class.

arcpy.management.XYTableToPoint(
    in_table=MuseumCSV1,
    out_feature_class="MDF1",
    x_field="LONGITUDE",
    y_field="LATITUDE",
    z_field=None,
    coordinate_system='GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SPHEROID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]];-400 -400 1000000000;-100000 10000;-100000 10000;8.98315284119521E-09;0.001;0.001;IsHighPrecision'
)
arcpy.management.XYTableToPoint(
    in_table=MuseumCSV2,
    out_feature_class="MDF2",
    x_field="LONGITUDE",
    y_field="LATITUDE",
    z_field=None,
    coordinate_system='GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SPHEROID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]];-400 -400 1000000000;-100000 10000;-100000 10000;8.98315284119521E-09;0.001;0.001;IsHighPrecision'
)
arcpy.management.XYTableToPoint(
    in_table=MuseumCSV3,
    out_feature_class="MDF3",
    x_field="LONGITUDE",
    y_field="LATITUDE",
    z_field=None,
    coordinate_system='GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SPHEROID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]];-400 -400 1000000000;-100000 10000;-100000 10000;8.98315284119521E-09;0.001;0.001;IsHighPrecision'
)
MDF_File_Path = os.path.join(workspace, "MDFMuseums")
arcpy.management.Merge(["MDF1", "MDF2", "MDF3"], MDF_File_Path)

<class 'NameError'>: name 'MDF1' is not defined

In [ ]:
#Extract Multi Values to Points to sample SLR Inundation values at the locations of MDFMuseums 
arcpy.sa.ExtractMultiValuesToPoints("MDFMuseums", [[SLR_High_Tide, "SLR_High_Tide"], 
                           [SLR_EC_Int, "SLR_EC_Int"], [SLR_LC_Intermediate, "SLR_LC_Intermediate"], [SLR_LC_IntHigh, "SLR_LC_IntHigh"]], "NONE")

# NEW CODE, NEED TO ADJUST

In [2]:

#Subset multidimensional raster to just the SSP 2-4.5 projections
arcpy.md.SubsetMultidimensionalRaster(
    in_multidimensional_raster=TemperatureMultidimensional,
    out_multidimensional_raster=r"SSP245_Subset.crf",
    variables="Annual_number_days_max_temp_GE_90F;Cooling_Degree_Days;Heating_Degree_Days",
    dimension_def="BY_RANGES",
    dimension_ranges="SSP 245 245",
    dimension_values=None,
    dimension="",
    start_of_first_iteration="",
    end_of_first_iteration="",
    iteration_step=None,
    iteration_unit=""
)

#Aggregate raster for each scenario - Early Century (2016-2045) and Late Century (2070-2100) for Days over 90, CDD, and HDD
with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Annual_number_days_max_temp_GE_90F",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2016-12-31T00:00:00 2045-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Days90_EC_SSP245.crf")

with arcpy.EnvManager(scratchWorkspace=r"C:\Users\mdh51\Documents\ArcGIS\Projects\Museums\Museums.gdb"):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Annual_number_days_max_temp_GE_90F",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-12-31T00:00:00 2100-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Days90_LC_SSP245.crf")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Cooling_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2016-12-31T00:00:00 2045-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"CDD_EC_SSP245.crf")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Cooling_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-12-31T00:00:00 2100-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"CDD_LC_SSP245.crf")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Heating_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2016-12-31T00:00:00 2045-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"HDD_EC_SSP245.crf")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Heating_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-12-31T00:00:00 2100-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"HDD_LC_SSP245.crf")

#Aggregate historic temperature variables from 1950-1980

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Annual_number_days_max_temp_GE_90F",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="1950-12-31T00:00:00 1980-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Days90_Historic.crf")

print("done Days90_Historic")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Cooling_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="1950-12-31T00:00:00 1980-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"CDD_Historic.crf")

print("done CDD_Historic")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Heating_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="1950-12-31T00:00:00 1980-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"HDD_Historic.crf")

print("done HDD_Historic")
print("all done!")

In [2]:
#Calculating Minimums and Maximums for Days over 90, CDD, and HDD for SSP2-4.5
#Aggregate raster for each scenario - Early Century (2016-2045) and Late Century (2070-2100) for Days over 90, CDD, and HDD
with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Annual_number_days_max_temp_GE_90F",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2016-12-31T00:00:00 2045-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_Days90_EC_SSP245.crf")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Annual_number_days_max_temp_GE_90F",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2016-12-31T00:00:00 2045-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_Days90_EC_SSP245.crf")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Annual_number_days_max_temp_GE_90F",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-12-31T00:00:00 2100-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_Days90_LC_SSP245.crf")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Annual_number_days_max_temp_GE_90F",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-12-31T00:00:00 2100-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_Days90_LC_SSP245.crf")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Cooling_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2016-12-31T00:00:00 2045-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_CDD_EC_SSP245.crf")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Cooling_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2016-12-31T00:00:00 2045-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_CDD_EC_SSP245.crf")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Cooling_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-12-31T00:00:00 2100-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_CDD_LC_SSP245.crf")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Cooling_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-12-31T00:00:00 2100-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_CDD_LC_SSP245.crf")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Heating_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2016-12-31T00:00:00 2045-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_HDD_EC_SSP245.crf")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Heating_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2016-12-31T00:00:00 2045-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_HDD_EC_SSP245.crf")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Heating_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-12-31T00:00:00 2100-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_HDD_LC_SSP245.crf")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Heating_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-12-31T00:00:00 2100-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_HDD_LC_SSP245.crf")

#Aggregate historic temperature variables from 1950-1980

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=r"C:\Users\mdh51\Documents\ArcGIS\Projects\Museums\SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Annual_number_days_max_temp_GE_90F",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="1950-12-31T00:00:00 1980-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_Days90_Historic.crf")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=r"C:\Users\mdh51\Documents\ArcGIS\Projects\Museums\SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Annual_number_days_max_temp_GE_90F",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="1950-12-31T00:00:00 1980-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_Days90_Historic.crf")

print("done Days90_Historic")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=r"C:\Users\mdh51\Documents\ArcGIS\Projects\Museums\SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Cooling_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="1950-12-31T00:00:00 1980-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_CDD_Historic.crf")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=r"C:\Users\mdh51\Documents\ArcGIS\Projects\Museums\SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Cooling_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="1950-12-31T00:00:00 1980-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_CDD_Historic.crf")

print("done CDD_Historic")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=r"C:\Users\mdh51\Documents\ArcGIS\Projects\Museums\SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Heating_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="1950-12-31T00:00:00 1980-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_HDD_Historic.crf")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=r"C:\Users\mdh51\Documents\ArcGIS\Projects\Museums\SSP245_Subset.crf",
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Heating_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="1950-12-31T00:00:00 1980-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_HDD_Historic.crf")

print("done HDD_Historic")
print("all done!")

done Days90_Historic
done CDD_Historic
done HDD_Historic
all done!


In [3]:
#Calculating Minimums and Maximums for Days over 90, CDD, and HDD for SSP3-7.0
#Aggregate raster for each scenario - Early Century (2016-2045) and Late Century (2070-2100) for Days over 90, CDD, and HDD

inraster = r"SSP370_Subset.crf"

arcpy.md.SubsetMultidimensionalRaster(
    in_multidimensional_raster=TemperatureMultidimensional,
    out_multidimensional_raster=r"SSP370_Subset.crf",
    variables="Annual_number_days_max_temp_GE_90F;Cooling_Degree_Days;Heating_Degree_Days",
    dimension_def="BY_RANGES",
    dimension_ranges="SSP 370 370",
    dimension_values=None,
    dimension="",
    start_of_first_iteration="",
    end_of_first_iteration="",
    iteration_step=None,
    iteration_unit=""
)

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=inraster,
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Annual_number_days_max_temp_GE_90F",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-12-31T00:00:00 2100-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_Days90_LC_SSP370.crf")

print("done Min_Days90_LC_SSP370")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=inraster,
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Annual_number_days_max_temp_GE_90F",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-12-31T00:00:00 2100-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_Days90_LC_SSP370.crf")

print("done Max_Days90_LC_SSP370")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=inraster,
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Cooling_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-12-31T00:00:00 2100-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_CDD_LC_SSP370.crf")

print("done Min_CDD_LC_SSP370")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=inraster,
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Cooling_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-12-31T00:00:00 2100-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_CDD_LC_SSP370.crf")

print("done Max_CDD_LC_SSP370")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=inraster,
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Heating_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-12-31T00:00:00 2100-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_HDD_LC_SSP370.crf")

print("done Min_HDD_LC_SSP370")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=inraster,
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Heating_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-12-31T00:00:00 2100-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_HDD_LC_SSP370.crf")

print("done Max_HDD_LC_SSP370")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP370_Subset.crf",
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Annual_number_days_max_temp_GE_90F",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-12-31T00:00:00 2100-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Days90_LC_SSP370.crf")

print("done Days90_LC_SSP370")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP370_Subset.crf",
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Cooling_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-12-31T00:00:00 2100-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"CDD_LC_SSP370.crf")

print("done CDD_LC_SSP370")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster="SSP370_Subset.crf",
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Heating_Degree_Days",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-12-31T00:00:00 2100-12-31T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"HDD_LC_SSP370.crf")

print("done HDD_LC_SSP370")
print("all done!")

done Min_Days90_LC_SSP370
done Max_Days90_LC_SSP370
done Min_CDD_LC_SSP370
done Max_CDD_LC_SSP370
done Min_HDD_LC_SSP370
done Max_HDD_LC_SSP370
all done!


In [5]:
#Aggregate raster for each scenario - Historic, (1950-1980), Early Century (2016-2045) and Late Century (2070-2100) for precipitation variables
#for Means

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip370Multidimensional,
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Annual_dry_spell_max_length",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-01-01T00:00:00 2100-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"C:\Users\mdh51\Documents\ArcGIS\Projects\Museums\DryDays_LC_SSP370.crf")

print("done DryDays_LC_SSP370")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip370Multidimensional,
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Annual_total_precip",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-01-01T00:00:00 2100-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"C:\Users\mdh51\Documents\ArcGIS\Projects\Museums\Precip_LC_SSP370.crf")

print("done TotalPrecip_LC_SSP370")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip370Multidimensional,
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Annual_pr_days_above_nonzero_99th",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-01-01T00:00:00 2100-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"C:\Users\mdh51\Documents\ArcGIS\Projects\Museums\Precip99_LC_SSP370.crf")

print("done Precip99_LC_SSP370")

#### LC SSP2-4.5

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip370Multidimensional,
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Annual_dry_spell_max_length",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-01-01T00:00:00 2100-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"C:\Users\mdh51\Documents\ArcGIS\Projects\Museums\DryDays_LC_SSP245.crf")

print("done DryDays_LC_SSP245")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip370Multidimensional,
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Annual_total_precip",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-01-01T00:00:00 2100-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Precip_LC_SSP245.crf")

print("done DryDays_LC_SSP245")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Annual_pr_days_above_nonzero_99th",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-01-01T00:00:00 2100-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Precip99_LC_SSP245.crf")

print("done DryDays_LC_SSP245")

#### EC SSP2-4.5

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Annual_dry_spell_max_length",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2016-01-01T00:00:00 2045-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"DryDays_EC_SSP245.crf")

print("done DryDays_EC_SSP245")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Annual_total_precip",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2016-01-01T00:00:00 2045-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Precip_EC_SSP245.crf")

print("done DryDays_EC_SSP245")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Annual_pr_days_above_nonzero_99th",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2016-01-01T00:00:00 2045-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Precip99_EC_SSP245.crf")

print("done DryDays_EC_SSP245")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Annual_dry_spell_max_length",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="1950-01-01T00:00:00 1980-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"DryDays_Historic.crf")

print("done DryDays_Historic")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Annual_total_precip",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="1950-01-01T00:00:00 1980-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Precip_Historic.crf")

print("done Precip_Historic")

#Historic variables below - historic range is 1950-1980 according to CRIS

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Annual_pr_days_above_nonzero_99th",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="1950-01-01T00:00:00 1980-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Precip99_Historic.crf")

print("done Precip99_Historic")
print("all done")

done DryDays_LC_SSP370
done TotalPrecip_LC_SSP370
done Precip99_LC_SSP370
done DryDays_LC_SSP245
done DryDays_LC_SSP245
done DryDays_LC_SSP245
done DryDays_EC_SSP245
done DryDays_EC_SSP245
done DryDays_EC_SSP245
all done


In [3]:
#Aggregate raster for each scenario - Historic, (1950-1980), Early Century (2016-2045) and Late Century (2070-2100) for precipitation variables
#for Min and Max

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip370Multidimensional,
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Annual_dry_spell_max_length",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-01-01T00:00:00 2100-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_DryDays_LC_SSP370.crf")

print("done Max_DryDays_LC_SSP370")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip370Multidimensional,
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Annual_dry_spell_max_length",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-01-01T00:00:00 2100-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_DryDays_LC_SSP370.crf")

print("done Min_DryDays_LC_SSP370")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip370Multidimensional,
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Annual_total_precip",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-01-01T00:00:00 2100-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_Precip_LC_SSP370.crf")

print("done Max_Precip_LC_SSP370")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip370Multidimensional,
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Annual_total_precip",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-01-01T00:00:00 2100-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_Precip_LC_SSP370.crf")

print("done Min_Precip_LC_SSP370")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip370Multidimensional,
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Annual_pr_days_above_nonzero_99th",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-01-01T00:00:00 2100-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_Precip99_LC_SSP370.crf")

print("done Max_Precip99_LC_SSP370")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip370Multidimensional,
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Annual_pr_days_above_nonzero_99th",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-01-01T00:00:00 2100-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_Precip99_LC_SSP370.crf")

print("done Min_Precip99_LC_SSP370")

#### LC SSP2-4.5

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Annual_dry_spell_max_length",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-01-01T00:00:00 2100-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_DryDays_LC_SSP245.crf")

print("done Max_DryDays_LC_SSP245")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Annual_dry_spell_max_length",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-01-01T00:00:00 2100-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_DryDays_LC_SSP245.crf")

print("done Min_DryDays_LC_SSP245")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Annual_total_precip",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-01-01T00:00:00 2100-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_Precip_LC_SSP245.crf")

print("done Max_Precip_LC_SSP245")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Annual_total_precip",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-01-01T00:00:00 2100-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_Precip_LC_SSP245.crf")

print("done Min_Precip_LC_SSP245")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Annual_pr_days_above_nonzero_99th",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-01-01T00:00:00 2100-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_Precip99_LC_SSP245.crf")

print("done Max_Precip99_LC_SSP245")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Annual_pr_days_above_nonzero_99th",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2070-01-01T00:00:00 2100-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_Precip99_LC_SSP245.crf")

print("done Min_Precip99_LC_SSP245")

#### EC SSP2-4.5

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Annual_dry_spell_max_length",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2016-01-01T00:00:00 2045-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_DryDays_EC_SSP245.crf")

print("done Max_DryDays_EC_SSP245")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Annual_dry_spell_max_length",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2016-01-01T00:00:00 2045-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_DryDays_EC_SSP245.crf")

print("done Min_DryDays_EC_SSP245")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Annual_total_precip",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2016-01-01T00:00:00 2045-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_Precip_EC_SSP245.crf")

print("done Max_Precip_EC_SSP245")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Annual_total_precip",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2016-01-01T00:00:00 2045-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_Precip_EC_SSP245.crf")

print("done Min_Precip_EC_SSP245")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Annual_pr_days_above_nonzero_99th",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2016-01-01T00:00:00 2045-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_Precip99_EC_SSP245.crf")

print("done Max_Precip99_EC_SSP245")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Annual_pr_days_above_nonzero_99th",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="2016-01-01T00:00:00 2045-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_Precip99_EC_SSP245.crf")

print("done Min_Precip99_EC_SSP245")

#Historic variables below - historic range is 1950-1980 according to CRIS

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Annual_dry_spell_max_length",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="1950-01-01T00:00:00 1980-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_DryDays_Historic.crf")

print("done Max_DryDays_Historic")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Annual_dry_spell_max_length",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="1950-01-01T00:00:00 1980-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_DryDays_Historic.crf")

print("done Min_DryDays_Historic")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Annual_total_precip",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="1950-01-01T00:00:00 1980-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_Precip_Historic.crf")

print("done Max_Precip_Historic")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Annual_total_precip",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="1950-01-01T00:00:00 1980-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_Precip_Historic.crf")

print("done Min_Precip_Historic")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MAXIMUM",
        variables="Annual_pr_days_above_nonzero_99th",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="1950-01-01T00:00:00 1980-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Max_Precip99_Historic.crf")

print("done Max_Precip99_Historic")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MINIMUM",
        variables="Annual_pr_days_above_nonzero_99th",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="1950-01-01T00:00:00 1980-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Min_Precip99_Historic.crf")

print("done Min_Precip99_Historic")
print("all done")

done Max_DryDays_LC_SSP370
done Min_DryDays_LC_SSP370
done Max_Precip_LC_SSP370
done Min_Precip_LC_SSP370
done Max_Precip99_LC_SSP370
done Min_Precip99_LC_SSP370
done Max_DryDays_LC_SSP245
done Min_DryDays_LC_SSP245
done Max_Precip_LC_SSP245
done Min_Precip_LC_SSP245
done Max_Precip99_LC_SSP245
done Min_Precip99_LC_SSP245
done Max_DryDays_EC_SSP245
done Min_DryDays_EC_SSP245
done Max_Precip_EC_SSP245
done Min_Precip_EC_SSP245
done Max_Precip99_EC_SSP245
done Min_Precip99_EC_SSP245
done Max_DryDays_Historic
done Min_DryDays_Historic
done Max_Precip_Historic
done Min_Precip_Historic
done Max_Precip99_Historic
done Min_Precip99_Historic
all done


In [2]:
#Aggregate raster for historic data scenario for precipitation variables

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Annual_dry_spell_max_length",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="1950-01-01T00:00:00 1980-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"DryDays_Historic.crf")

print("done DryDays_Historic")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Annual_total_precip",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="1950-01-01T00:00:00 1980-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Precip_Historic.crf")

print("done Precip_Historic")

with arcpy.EnvManager(scratchWorkspace=workspace):
    out_multidimensional_raster = arcpy.ia.AggregateMultidimensionalRaster(
        in_multidimensional_raster=Precip245Multidimensional,
        dimension="StdTime",
        aggregation_method="MEAN",
        variables="Annual_pr_days_above_nonzero_99th",
        aggregation_def="INTERVAL_RANGES",
        interval_keyword="",
        interval_value=None,
        interval_unit="",
        interval_ranges="1950-01-01T00:00:00 1980-01-01T00:00:00",
        aggregation_function="",
        ignore_nodata="DATA",
        dimensionless="DIMENSIONS",
        percentile_value=90,
        percentile_interpolation_type="NEAREST"
    )
    out_multidimensional_raster.save(r"Precip99_Historic.crf")

print("done Precip99_Historic")


print("all done")

done DryDays_Historic
done Precip_Historic
all done


In [1]:
#extract multi values to points for precipitation related data

arcpy.sa.ExtractMultiValuesToPoints(
    in_point_features="MDFMuseums",
    in_rasters=r"Precip99_EC_SSP245.crf Precip99_EC_SSP245;Precip_EC_SSP245.crf Precip_EC_SSP245;DryDays_EC_SSP245.crf DryDays_EC_SSP245;Precip99_LC_SSP245.crf Precip99_LC_SSP245;Precip_LC_SSP245.crf Precip_LC_SSP245;DryDays_LC_SSP245.crf DryDays_LC_SSP245;Precip99_LC_SSP370.crf Precip99_LC_SSP370;Precip_LC_SSP370.crf Precip_LC_SSP370;DryDays_LC_SSP370.crf DryDays_LC_SSP370",
    bilinear_interpolate_values="NONE"
)

<Result 'MDFMuseums'>

In [6]:
#extract multi values to points for CDD, HDD, and Days over 90 medians for SSP2-4.5
arcpy.sa.ExtractMultiValuesToPoints(
    in_point_features="MDF Museums",
    in_rasters="Days90_LC_SSP370.crf D90_LC_370;Days90_LC_SSP245.crf D90_LC_245;Days90_EC_SSP245.crf D90_EC_245;Historic90days.crf D90_Historic;CDD_LC_SSP370.crf CDD_LC_370;CDD_LC_SSP245.crf CDD_LC_245;CDD_EC_SSP245.crf CDD_EC_245;CDD_Historic.crf CDD_Historic;HDD_LC_SSP370.crf HDD_LC_370;HDD_LC_SSP245.crf HDD_LC_245;HDD_EC_SSP245.crf HDD_EC_245;HDD_Historic.crf HDD_Historic",
    bilinear_interpolate_values="NONE"
)

#extract multi values to points for CDD, HDD, and Days over 90 mins and maxes for SSP2-4.5
arcpy.sa.ExtractMultiValuesToPoints(
    in_point_features="MDFMuseums",
    in_rasters="Min_HDD_LC_SSP245.crf Min_HDD_LC_SSP245;Min_HDD_Historic.crf Min_HDD_Historic;Min_HDD_EC_SSP245.crf Min_HDD_EC_SSP245;Min_Days90_LC_SSP245.crf Min_Days90_LC_SSP245;Min_Days90_Historic.crf Min_Days90_Historic;Min_Days90_EC_SSP245.crf Min_Days90_EC_SSP245;Min_CDD_LC_SSP245.crf Min_CDD_LC_SSP245;Min_CDD_Historic.crf Min_CDD_Historic;Min_CDD_EC_SSP245.crf Min_CDD_EC_SSP245;Max_HDD_LC_SSP245.crf Max_HDD_LC_SSP245;Max_HDD_Historic.crf Max_HDD_Historic;Max_HDD_EC_SSP245.crf Max_HDD_EC_SSP245;Max_Days90_LC_SSP245.crf Max_Days90_LC_SSP245;Max_Days90_Historic.crf Max_Days90_Historic;Max_Days90_EC_SSP245.crf Max_Days90_EC_SSP245;Max_CDD_LC_SSP245.crf Max_CDD_LC_SSP245;Max_CDD_Historic.crf Max_CDD_Historic;Max_CDD_EC_SSP245.crf Max_CDD_EC_SSP245",
    bilinear_interpolate_values="NONE"
)

#extract multi values to points for CDD and HDD mins and maxes for SSP3-7.0


arcpy.sa.ExtractMultiValuesToPoints(
    in_point_features="MDFMuseums",
    in_rasters="Max_CDD_LC_SSP370.crf Max_CDD_LC_SSP370;Max_Days90_LC_SSP370.crf Max_Days90_LC_SSP370;Max_HDD_LC_SSP370.crf Max_HDD_LC_SSP370;Min_CDD_LC_SSP370.crf Min_CDD_LC_SSP370;Min_Days90_LC_SSP370.crf Min_Days90_LC_SSP370;Min_HDD_LC_SSP370.crf Min_HDD_LC_SSP370",
    bilinear_interpolate_values="NONE"
)


In [36]:
flood_raster_string = "https://landscape11.arcgis.com/arcgis/services/USA_Flood_Hazard_Areas/ImageServer"

with arcpy.EnvManager(extent='1554456.2939096 584399.361169408 1582746.66635745 600079.304986394 PROJCRS["USA_Contiguous_Albers_Equal_Area_Conic_USGS_version",BASEGEOGCRS["GCS_North_American_1983",DATUM["D_North_American_1983",ELLIPSOID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433]],CONVERSION["Albers",METHOD["Albers"],PARAMETER["False_Easting",0.0],PARAMETER["False_Northing",0.0],PARAMETER["Central_Meridian",-96.0],PARAMETER["Standard_Parallel_1",29.5],PARAMETER["Standard_Parallel_2",45.5],PARAMETER["Latitude_Of_Origin",23.0]],CS[Cartesian,2],AXIS["Easting (X)",east,ORDER[1]],AXIS["Northing (Y)",north,ORDER[2]],LENGTHUNIT["Meter",1.0],ID["Esri","102039"]]'):
    arcpy.sa.ExtractMultiValuesToPoints("MDFMuseums", [[flood_raster_string, "floodingtest"]], "NONE")

In [ ]:
#Create grid to divide CONUS into smaller pieces, then interate over each grid square applying Extract Multi Values to Points on FEMA Flood Values

arcpy.management.GenerateTessellation(
    Output_Feature_Class="GenerateTessellation2",
    Extent='-3336830.29706174 -67562.6395372031 2916831.99492706 4109323.74205233 PROJCS["USA_Contiguous_Albers_Equal_Area_Conic_USGS_version",GEOGCS["GCS_North_American_1983",DATUM["D_North_American_1983",SPHEROID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Albers"],PARAMETER["False_Easting",0.0],PARAMETER["False_Northing",0.0],PARAMETER["Central_Meridian",-96.0],PARAMETER["Standard_Parallel_1",29.5],PARAMETER["Standard_Parallel_2",45.5],PARAMETER["Latitude_Of_Origin",23.0],UNIT["Meter",1.0]]',
    Shape_Type="SQUARE",
    Size="5625000000 SquareMeters",
    Spatial_Reference='PROJCS["USA_Contiguous_Albers_Equal_Area_Conic_USGS_version",GEOGCS["GCS_North_American_1983",DATUM["D_North_American_1983",SPHEROID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Albers"],PARAMETER["False_Easting",0.0],PARAMETER["False_Northing",0.0],PARAMETER["Central_Meridian",-96.0],PARAMETER["Standard_Parallel_1",29.5],PARAMETER["Standard_Parallel_2",45.5],PARAMETER["Latitude_Of_Origin",23.0],UNIT["Meter",1.0]];-16901100 -6972200 10000;-100000 10000;-100000 10000;0.001;0.001;0.001;IsHighPrecision',
    H3_Resolution=7
)

# Inputs
tess = "GenerateTessellation2"
points = "MDFMuseums"
flood_raster_string = "https://landscape11.arcgis.com/arcgis/services/USA_Flood_Hazard_Areas/ImageServer"

small_grid_ids = []
# Iterate through only the selected features
with arcpy.da.SearchCursor(tess, ["GRID_ID"]) as cursor:
    for row in cursor:
        small_grid_ids.append(row[0])

print("GRID_ID values from selected features:", small_grid_ids)

# Create an  field to hold running totals
accum_field = "FloodVal"
fields = [f.name for f in arcpy.ListFields(points)]
if accum_field not in fields:
    arcpy.AddField_management(points, accum_field, "SHORT")

for i, grid_id in enumerate(small_grid_ids, start=1):
    # Select polygon by GRID_ID
    where_clause = f"GRID_ID = '{grid_id}'"
    arcpy.management.SelectLayerByAttribute(tess, "NEW_SELECTION", where_clause)

    # Get extent of selected feature
    with arcpy.da.SearchCursor(tess, ["SHAPE@"]) as cursor:
        for row in cursor:
            extent = row[0].extent
            arcpy.env.extent = extent
            print(f"Processing GRID_ID {grid_id} with extent {extent}")

            # Name a temp field for this iteration
            temp_field = f"FloodVal_tmp{i}"

            # Extract values into the temporary field
            arcpy.sa.ExtractMultiValuesToPoints(
                in_point_features=points,
                in_rasters=[[flood_raster_string, temp_field]],
                bilinear_interpolate_values="NONE"
            )

            # Update the accumulator field by adding the temporary field values
            with arcpy.da.UpdateCursor(points, [accum_field, temp_field]) as ucur:
                for urow in ucur:
                    if urow[1] is not None:
                        urow[0] = urow[1]   # Replace instead of accumulate
                        ucur.updateRow(urow)

            # Delete the temporary field
            arcpy.DeleteField_management(points, [temp_field])

    arcpy.env.extent = None

In [ ]:
#join table from https://www.arcgis.com/home/item.html?id=11955f1b47ec41a3af86650824e0c634#overview to museum point feature class based on sampled Flood Value field

arcpy.management.JoinField(
    in_data="MDF Museums",
    in_field="FloodVal",
    join_table="FloodZoneTable.csv",
    join_field="Value",
    fields=None,
    fm_option="USE_FM",
    field_mapping='FLD_ZONE "Flood Zone" true true false 8000 Text 0 0,First,#,FloodZoneTable.csv,FLD_ZONE,0,7999;ZONE_SUBTY "Flood Zone Subtype" true true false 8000 Text 0 0,First,#,FloodZoneTable.csv,ZONE_SUBTY,0,7999;SFHA_TF "FHA - True/False" true true false 8000 Text 0 0,First,#,FloodZoneTable.csv,SFHA_TF,0,7999',
    index_join_fields="NO_INDEXES"
)

In [1]:
arcpy.management.ProjectRaster(
    in_raster=r"SLOSH MOMs\us_Category5_MOM_Inundation_HIGH.tif",
    out_raster=r"SLOSH_MOM_Cat5.tif",
    out_coor_system='GEOGCRS["GCS_WGS_1984",DYNAMIC[FRAMEEPOCH[1990.5],MODEL["AM0-2"]],DATUM["D_WGS_1984",ELLIPSOID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433],ID["EPSG","4326"]]',
    resampling_type="NEAREST",
    cell_size="9.78922489999944E-05 9.78922489999945E-05",
    geographic_transform="WGS_1984_(ITRF00)_To_NAD_1983",
    Registration_Point=None,
    in_coor_system='GEOGCRS["GCS_North_American_1983",DATUM["D_North_American_1983",ELLIPSOID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433],ID["EPSG","4269"]]',
    vertical="NO_VERTICAL"
)
arcpy.management.ProjectRaster(
    in_raster=r"SLOSH MOMs\us_Category4_MOM_Inundation_HIGH.tif",
    out_raster=r"SLOSH_MOM_Cat4.tif",
    out_coor_system='GEOGCRS["GCS_WGS_1984",DYNAMIC[FRAMEEPOCH[1990.5],MODEL["AM0-2"]],DATUM["D_WGS_1984",ELLIPSOID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433],ID["EPSG","4326"]]',
    resampling_type="NEAREST",
    cell_size="9.78922489999944E-05 9.78922489999945E-05",
    geographic_transform="WGS_1984_(ITRF00)_To_NAD_1983",
    Registration_Point=None,
    in_coor_system='GEOGCRS["GCS_North_American_1983",DATUM["D_North_American_1983",ELLIPSOID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433],ID["EPSG","4269"]]',
    vertical="NO_VERTICAL"
)
arcpy.management.ProjectRaster(
    in_raster=r"SLOSH MOMs\us_Category3_MOM_Inundation_HIGH.tif",
    out_raster=r"SLOSH_MOM_Cat3.tif",
    out_coor_system='GEOGCRS["GCS_WGS_1984",DYNAMIC[FRAMEEPOCH[1990.5],MODEL["AM0-2"]],DATUM["D_WGS_1984",ELLIPSOID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433],ID["EPSG","4326"]]',
    resampling_type="NEAREST",
    cell_size="9.78922489999944E-05 9.78922489999945E-05",
    geographic_transform="WGS_1984_(ITRF00)_To_NAD_1983",
    Registration_Point=None,
    in_coor_system='GEOGCRS["GCS_North_American_1983",DATUM["D_North_American_1983",ELLIPSOID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433],ID["EPSG","4269"]]',
    vertical="NO_VERTICAL"
)
arcpy.management.ProjectRaster(
    in_raster=r"SLOSH MOMs\us_Category2_MOM_Inundation_HIGH.tif",
    out_raster=r"SLOSH_MOM_Cat2.tif",
    out_coor_system='GEOGCRS["GCS_WGS_1984",DYNAMIC[FRAMEEPOCH[1990.5],MODEL["AM0-2"]],DATUM["D_WGS_1984",ELLIPSOID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433],ID["EPSG","4326"]]',
    resampling_type="NEAREST",
    cell_size="9.78922489999944E-05 9.78922489999945E-05",
    geographic_transform="WGS_1984_(ITRF00)_To_NAD_1983",
    Registration_Point=None,
    in_coor_system='GEOGCRS["GCS_North_American_1983",DATUM["D_North_American_1983",ELLIPSOID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433],ID["EPSG","4269"]]',
    vertical="NO_VERTICAL"
)
arcpy.management.ProjectRaster(
    in_raster=r"SLOSH MOMs\us_Category1_MOM_Inundation_HIGH.tif",
    out_raster=r"SLOSH_MOM_Cat1.tif",
    out_coor_system='GEOGCRS["GCS_WGS_1984",DYNAMIC[FRAMEEPOCH[1990.5],MODEL["AM0-2"]],DATUM["D_WGS_1984",ELLIPSOID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433],ID["EPSG","4326"]]',
    resampling_type="NEAREST",
    cell_size="9.78922489999944E-05 9.78922489999945E-05",
    geographic_transform="WGS_1984_(ITRF00)_To_NAD_1983",
    Registration_Point=None,
    in_coor_system='GEOGCRS["GCS_North_American_1983",DATUM["D_North_American_1983",ELLIPSOID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433],ID["EPSG","4269"]]',
    vertical="NO_VERTICAL"
)

<Result 'C:\\Users\\mdh51\\OneDrive - Wichita State University\\Research\\Museum Project\\SLOSH_MOM_Cat1.tif'>

In [1]:
arcpy.management.ProjectRaster(
    in_raster=r"SLOSH MOMs\scp_Category1_MOM_Inundation_HIGH.tif",
    out_raster=r"SLOSH_MOM_Cat1_SCP.tif",
    out_coor_system='GEOGCRS["GCS_WGS_1984",DYNAMIC[FRAMEEPOCH[1990.5],MODEL["AM0-2"]],DATUM["D_WGS_1984",ELLIPSOID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433],ID["EPSG","4326"]]',
    resampling_type="NEAREST",
    cell_size="9.78922489999944E-05 9.78922489999945E-05",
    geographic_transform="WGS_1984_(ITRF00)_To_NAD_1983",
    Registration_Point=None,
    in_coor_system='GEOGCRS["GCS_North_American_1983",DATUM["D_North_American_1983",ELLIPSOID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433],ID["EPSG","4269"]]',
    vertical="NO_VERTICAL"
)
arcpy.management.ProjectRaster(
    in_raster=r"SLOSH MOMs\scp_Category2_MOM_Inundation_HIGH.tif",
    out_raster=r"SLOSH_MOM_Cat2_SCP.tif",
    out_coor_system='GEOGCRS["GCS_WGS_1984",DYNAMIC[FRAMEEPOCH[1990.5],MODEL["AM0-2"]],DATUM["D_WGS_1984",ELLIPSOID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433],ID["EPSG","4326"]]',
    resampling_type="NEAREST",
    cell_size="9.78922489999944E-05 9.78922489999945E-05",
    geographic_transform="WGS_1984_(ITRF00)_To_NAD_1983",
    Registration_Point=None,
    in_coor_system='GEOGCRS["GCS_North_American_1983",DATUM["D_North_American_1983",ELLIPSOID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433],ID["EPSG","4269"]]',
    vertical="NO_VERTICAL"
)

<Result 'C:\\Users\\mdh51\\OneDrive - Wichita State University\\Research\\Museum Project\\SLOSH_MOM_Cat2_SCP.tif'>

In [8]:
#Extract Multi Values to Points for Storm Surge raster layers, downloaded from https://www.nhc.noaa.gov/nationalsurge/?text
import os
data_dir1 = r"C:\Users\mdh51\OneDrive - Wichita State University\Research\Museum Project"

EastCat5 = os.path.join(data_dir1, "SLOSH_MOM_Cat5.tif")
EastCat4 = os.path.join(data_dir1, "SLOSH_MOM_Cat4.tif")
EastCat3 = os.path.join(data_dir1, "SLOSH_MOM_Cat3.tif")
EastCat2 = os.path.join(data_dir1, "SLOSH_MOM_Cat2.tif")
EastCat1 = os.path.join(data_dir1, "SLOSH_MOM_Cat1.tif")
WestCat2 = os.path.join(data_dir1, "SLOSH_MOM_Cat2_SCP.tif")
WestCat1 = os.path.join(data_dir1, "SLOSH_MOM_Cat1_SCP.tif")

arcpy.sa.ExtractMultiValuesToPoints("MDFMuseums", [[EastCat5, "EastCat5"], [EastCat4, "EastCat4"], [EastCat3, "EastCat3"], [EastCat2, "EastCat2"], [EastCat1, "EastCat1"], [WestCat2, "WestCat2"], [WestCat1, "WestCat1"]], "NONE")

<Result 'MDFMuseums'>

In [ ]:
#Extract Multi Values to Points to sample Wildfire Hazard Potential Inundation values at the locations of MDFMuseums 
arcpy.sa.ExtractMultiValuesToPoints("MDFMuseums", [WildfireHazardPotential, "WHP"], "NONE")